# GRPO-tuned fuzzy PID: proposed method and baselines

Run the cells in order. Every training cell skips runs that already have a
finished `_meta.npz` file, so after a crash just rerun the same cell.

1. Setup
2. Settings and GRPO functions
3. Plant-variation (DR) training function
4. Proposed method: type-1, fixed FOU (paper), tuned FOU
5. Baselines: Sahin & Ulu (2026), Kluska & Zabinski (2020)
6. Compare all methods

All methods use `antiwindup='clamp'`. Start in a folder **without** old
`.npy` / `.npz` files: file names do not record the anti-windup, so old files
would be skipped and mixed in.

## 1. Setup
CPU affinity for the worker pool.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"      # keep each worker single-threaded

import psutil
import multiprocessing as mp
from grpo_workers import init_worker, rollout

p = psutil.Process()
visible = p.cpu_affinity()
cpus = visible[::1][:4]                   # every other logical CPU = one per physical core
p.cpu_affinity(cpus)                      # worker processes inherit this
print(f"visible {len(visible)} logical CPUs -> using {len(cpus)}: {cpus}")

## 2. Settings and GRPO functions
GRPO hyperparameters, policy, update rule, `train`, `report`, `env_kw_for`.
`make_env` builds the proposed controller by default, or a baseline when the
env kwargs contain `method='sahin_ulu'` or `method='kluska'`.

In [ ]:
"""
GRPO tuning of the IT2 fuzzy PID scaling-factor corrections.

Bandit formulation: a single "state" (the plant), an action = the 6-dim vector
of log-scaled correction factors, one closed-loop rollout per action.

Differences from plain group-baseline REINFORCE:
  * old-policy importance ratio with PPO-style clipping,
  * several inner epochs per sampled group (sample reuse),
  * closed-form KL penalty to a FIXED reference policy centred on the nominal
    IMC design (mu = 0 -> all correction factors = 1).  This is the GRPO
    counterpart of anchoring the Bayesian search to the model-based nominal.
"""
import numpy as np
from mimo_fuzzy_env import PLANTS
from fuzzy_pid_t2 import FuzzyPIDT2Env
from baselines import make_env      # builds ours or a baseline

# ===================== settings =====================
PLANT     = 'vl'      # 'wb' | 'ww' | 'vl' | 'isp' | 'hvac' | 'chen'
MODE      = 'fou'       # 't1' | 'it2' (fixed FOU, as in the paper) | 'fou' (tune s_l too)
FT        = 1 if MODE == 't1' else 2
TUNE_FOU  = MODE == 'fou'
TR        = 'eiasc'     # 'eiasc' (exact KM, as MATLAB) | 'nietan' (fast)
TAUC_F    = 1.0         # tauC = TAUC_F * L   (paper: tauC ~= L)

# derivative-filter time constant; must satisfy tau_f >= 2*ts
TAU_F_OF  = {'hvac': 2.0, 'chen': 0.1}

def tau_f_for(plant, ts=None):
    ts = ts if ts is not None else PLANTS[plant]['ts']
    return max(TAU_F_OF.get(plant, 0.01), 4.0 * ts)

TAU_F = tau_f_for(PLANT)
N_ITERS   = 250        # GRPO iterations
G         = 16         # group size (rollouts per iteration)
# --- GRPO hyperparameters ---
EPOCHS    = 4           # inner epochs over each sampled group
CLIP_EPS  = 0.2         # ratio clipping
BETA_KL   = 0.02        # weight on KL(pi_theta || pi_ref)
LR        = 0.02        # Adam step size
INIT_LOGSTD = -0.7      # ~0.50 in pre-squash space
LOGSTD_MIN, LOGSTD_MAX = -2.5, 0.2
SEED      = 0
ANTIWINDUP = 'clamp'    # conditional integration, used by every method
# ====================================================

ENV_KW = dict(plant=PLANT, R_step=1.0, warn=False,
              fuzzy_type=FT, tr_method=TR, tauC_factor=TAUC_F, tau_f=TAU_F,
              tune_fou=TUNE_FOU, antiwindup=ANTIWINDUP)

# ------------------------------------------------------------------ policy
class GaussPolicy:
    """Diagonal Gaussian in pre-squash space; action = tanh(z)."""

    def __init__(self, dim, init_logstd=INIT_LOGSTD, seed=0):
        self.mu = np.zeros(dim)
        self.logstd = np.full(dim, float(init_logstd))
        self.rng = np.random.default_rng(seed)

    # ---- sampling
    def sample(self, n):
        z = self.mu + np.exp(self.logstd) * self.rng.standard_normal((n, len(self.mu)))
        return z, np.tanh(z)

    # ---- log-density and its gradients (per sample, per dim)
    def logp(self, z, mu=None, logstd=None):
        mu = self.mu if mu is None else mu
        logstd = self.logstd if logstd is None else logstd
        d = (z - mu) / np.exp(logstd)
        return (-0.5 * d ** 2 - logstd - 0.5 * np.log(2 * np.pi)).sum(axis=1)

    def dlogp(self, z):
        """d logp / d mu  and  d logp / d logstd, shape (n, dim) each."""
        std = np.exp(self.logstd)
        d = (z - self.mu) / std
        return d / std, d ** 2 - 1.0

    def params(self):
        return self.mu.copy(), self.logstd.copy()

def kl_to_ref(mu, logstd, mu_ref, logstd_ref):
    """KL( N(mu, std^2) || N(mu_ref, std_ref^2) ) and its gradients."""
    var, var_ref = np.exp(2 * logstd), np.exp(2 * logstd_ref)
    kl = (logstd_ref - logstd
          + (var + (mu - mu_ref) ** 2) / (2 * var_ref) - 0.5).sum()
    g_mu = (mu - mu_ref) / var_ref
    g_ls = var / var_ref - 1.0
    return kl, g_mu, g_ls

class Adam:
    def __init__(self, shape, lr, b1=0.9, b2=0.999, eps=1e-8):
        self.m = np.zeros(shape)
        self.v = np.zeros(shape)
        self.lr, self.b1, self.b2, self.eps, self.t = lr, b1, b2, eps, 0

    def step(self, g):
        self.t += 1
        self.m = self.b1 * self.m + (1 - self.b1) * g
        self.v = self.b2 * self.v + (1 - self.b2) * g ** 2
        mh = self.m / (1 - self.b1 ** self.t)
        vh = self.v / (1 - self.b2 ** self.t)
        return self.lr * mh / (np.sqrt(vh) + self.eps)

# ------------------------------------------------------------------ training
def grpo_update(pol, z, adv, mu_old, logstd_old, mu_ref, logstd_ref,
                opt_mu, opt_ls, epochs=EPOCHS, clip_eps=CLIP_EPS,
                beta_kl=BETA_KL):
    """Several clipped-surrogate epochs on one group of rollouts."""
    logp_old = pol.logp(z, mu_old, logstd_old)
    n = len(z)
    stats = dict(ratio=1.0, clipfrac=0.0, kl=0.0)

    for _ in range(epochs):
        logp = pol.logp(z)
        ratio = np.exp(np.clip(logp - logp_old, -20.0, 20.0))
        ratio_c = np.clip(ratio, 1.0 - clip_eps, 1.0 + clip_eps)

        # surrogate = mean( min(ratio*A, clip(ratio)*A) ); gradient flows only
        # through samples where the unclipped term is the active (smaller) one
        unclipped = ratio * adv
        clipped = ratio_c * adv
        active = (unclipped <= clipped).astype(float)

        g_mu_lp, g_ls_lp = pol.dlogp(z)                 # (n, dim)
        w = (active * ratio * adv)[:, None] / n         # d surrogate / d logp
        # ascent on the surrogate -> descent on its negative
        g_mu = -(w * g_mu_lp).sum(0)
        g_ls = -(w * g_ls_lp).sum(0)

        kl, k_mu, k_ls = kl_to_ref(pol.mu, pol.logstd, mu_ref, logstd_ref)
        g_mu += beta_kl * k_mu
        g_ls += beta_kl * k_ls

        pol.mu = pol.mu - opt_mu.step(g_mu)
        pol.logstd = np.clip(pol.logstd - opt_ls.step(g_ls),
                             LOGSTD_MIN, LOGSTD_MAX)

        stats = dict(ratio=float(ratio.mean()),
                     clipfrac=float(1.0 - active.mean()),
                     kl=float(kl))
    return stats

from grpo_workers import rollout

def train(n_iters=N_ITERS, G=G, lr=LR, seed=SEED, verbose=True, env_kw=None, pool=None):
    env = make_env(**(env_kw or ENV_KW))
    pol = GaussPolicy(env.n_act, seed=seed)

    # reference policy = initial policy = nominal IMC design (all k = 1)
    mu_ref, logstd_ref = pol.params()

    opt_mu, opt_ls = Adam(env.n_act, lr), Adam(env.n_act, lr)
    best = (-np.inf, env.nominal_action())
    hist = []

    for it in range(n_iters):
        mu_old, logstd_old = pol.params()
        z, acts = pol.sample(G)
        if pool is not None:
            rews = np.array(pool.map(rollout, list(acts)))      # parallel
        else:
            rews = np.array([env.rollout(a) for a in acts])     # serial

        j = int(np.argmax(rews))
        if rews[j] > best[0]:
            best = (float(rews[j]), acts[j].copy())

        adv = (rews - rews.mean()) / (rews.std() + 1e-8)

        st = grpo_update(pol, z, adv, mu_old, logstd_old, mu_ref, logstd_ref,
                         opt_mu, opt_ls)

        hist.append((it, rews.mean(), rews.max(), st['kl'], st['clipfrac']))
        if verbose and it % 10 == 0:
            print(f"it {it:4d}  mean {rews.mean():9.3f}  best {rews.max():9.3f}"
                  f"  |mu| {np.abs(pol.mu).max():.3f}"
                  f"  std {np.exp(pol.logstd).mean():.3f}"
                  f"  KLref {st['kl']:6.3f}  clip {st['clipfrac']:.2f}")

    return pol, np.array(hist), env, best

# ------------------------------------------------------------------ reporting
def evaluate(a, env_kw=None):
    env = make_env(**(env_kw or ENV_KW))
    return env.rollout(a), env.metrics(a), np.asarray(a)

def report(tag, a, env_kw=None):
    r, m, a = evaluate(a, env_kw)
    per = " / ".join(f"y{i+1} {v:.3f}" for i, v in enumerate(m['ise']))
    print(f"{tag:16s} reward {r:9.3f}   ISE {m['ise_tot']:9.3f}  "
          f"IAE {m['iae_tot']:9.3f}   (ISE {per})")
    return r, m, a

import os, time
import numpy as np
from mimo_fuzzy_env import PLANTS, fit_fopdt
from fuzzy_pid_t2 import FuzzyPIDT2Env

PLANTS_TO_RUN = ['vl', 'wb', 'ww', 'isp', 'hvac']
SKIP_DONE     = True      # skip plants already trained WITH the C0 fix (resume after a crash)

def env_kw_for(plant):
    return dict(plant=plant, R_step=1.0, warn=False, fuzzy_type=FT,
                tr_method=TR, tauC_factor=TAUC_F, tau_f=tau_f_for(plant),
                tune_fou=TUNE_FOU, antiwindup=ANTIWINDUP)

def check_c0_fix(plant='vl'):
    """Stop if the x2 fix on C0 is not active (stale module / no kernel restart)."""
    env = make_env(**env_kw_for(plant))
    K, T, L = fit_fopdt(*PLANTS[plant]['g'][0][0])
    ratio = env.ctl[0].equivalent_pid()[1] / (1.0 / (K * (TAUC_F * L + L / 2.0)))
    assert abs(ratio - 1.0) < 1e-6, \
        f"C0 fix not active (Ki ratio {ratio:.3f}): edit scaling_factors, restart kernel"
    print("C0 fix active: type-1 equivalent = full IMC-PID")

def already_done(fname):
    try:
        return int(np.load(fname + "_meta.npz", allow_pickle=True)['c0x2']) == 1
    except Exception:
        return False

## 3. Plant-variation (DR) training function
`train_dr`: one random plant within ±15% per iteration, shared by the whole
group; the mean policy is checked every 10 iterations on nominal + 8 fixed
validation plants and the best one is kept.

In [ ]:
from grpo_workers import rollout_on

DELTA     = 0.15                        # sampled range: +-15%
VAL_EVERY = 10                          # check the mean policy every 10 iterations
VAL_PLANTS = [None] + list(range(1000, 1008))   # nominal + 8 fixed plants

def validate(a, pool):
    return float(np.mean(pool.map(rollout_on, [(a, DELTA, s) for s in VAL_PLANTS])))

def train_dr(n_iters, env_kw, seed, pool, verbose=True):
    env = make_env(**env_kw)
    pol = GaussPolicy(env.n_act, seed=seed)
    mu_ref, logstd_ref = pol.params()
    opt_mu, opt_ls = Adam(env.n_act, LR), Adam(env.n_act, LR)

    plant_rng = np.random.default_rng(10_000 + seed)   # same plants for all methods
    a0 = np.tanh(pol.mu).copy()
    best_val, best_a, best_it = validate(a0, pool), a0, 0
    hist, val_hist = [], [(0, best_val)]

    for it in range(n_iters):
        mu_old, logstd_old = pol.params()
        z, acts = pol.sample(G)

        # one plant per iteration, shared by the whole group
        plant_seed = int(plant_rng.integers(2 ** 31))
        rews = np.array(pool.map(rollout_on,
                                 [(a, DELTA, plant_seed) for a in acts]))
        adv = (rews - rews.mean()) / (rews.std() + 1e-8)
        st = grpo_update(pol, z, adv, mu_old, logstd_old, mu_ref, logstd_ref,
                         opt_mu, opt_ls)
        hist.append((it, rews.mean(), rews.max(), st['kl'], st['clipfrac']))

        if (it + 1) % VAL_EVERY == 0 or it + 1 == n_iters:
            a_mu = np.tanh(pol.mu).copy()
            r_val = validate(a_mu, pool)
            val_hist.append((it + 1, r_val))
            if r_val > best_val:
                best_val, best_a, best_it = r_val, a_mu, it + 1
            if verbose:
                print(f"it {it + 1:4d}  group mean {rews.mean():9.3f}  "
                      f"val {r_val:9.3f}  best val {best_val:9.3f} (it {best_it})")

    return pol, np.array(hist), np.array(val_hist), best_a, best_val, best_it

## 4. Proposed method

Three controllers, seeds 0–4, all plants:
* `t1`  – type-1 fuzzy PID (3 values per loop)
* `it2` – IT2 with the fixed FOU of the paper (3 values per loop)
* `fou` – IT2 with tuned lower membership scales (7 values per loop)

### 4a. Nominal-plant training
Files: `a_fis_{plant}_{tag}_seed{seed}.npy`

In [ ]:
import time
import multiprocessing as mp
from grpo_workers import init_worker, rollout

SEEDS     = [0, 1, 2, 3, 4]
N_WORKERS = len(cpus)
RUNS = [('it2', 2, False),   # paper's method: fixed FOU, 3n values
        ('t1',  1, False),   # type-1: 3n values
        ('fou', 2, True)]    # tuned FOU: 7n values

def make_kw(plant, ft, tune):
    return {**env_kw_for(plant), 'fuzzy_type': ft, 'tune_fou': tune}

def run_file(plant, tag, seed):
    return f"a_fis_{plant}_{tag}_seed{seed}"

def is_done(fname):
    """Finished, and trained with the current N_ITERS."""
    try:
        meta = np.load(fname + "_meta.npz", allow_pickle=True)
        return int(meta['c0x2']) == 1 and len(meta['hist']) == N_ITERS
    except Exception:
        return False

print(f"N_ITERS = {N_ITERS}, inner EPOCHS = {EPOCHS}, G = {G}")
check_c0_fix()
ctx = mp.get_context('spawn')

for seed in SEEDS:
    for tag, ft, tune in RUNS:
        for plant in PLANTS_TO_RUN:
            fname = run_file(plant, tag, seed)
            if is_done(fname):
                print(f"[{tag} {plant} seed {seed}] already trained, skipping")
                continue

            print(f"\n=== {tag}: {PLANTS[plant]['name']}, seed {seed} ===")
            kw = make_kw(plant, ft, tune)
            env0 = make_env(**kw)
            t0 = time.time()

            with ctx.Pool(processes=N_WORKERS, initializer=init_worker,
                          initargs=(kw,)) as pool:
                a0 = env0.nominal_action()
                assert abs(pool.map(rollout, [a0])[0] - env0.rollout(a0)) < 1e-9
                pol, hist, _, best = train(n_iters=N_ITERS, env_kw=kw,
                                           seed=seed, pool=pool)

            r_mu, _, a_mu = report("tuned (mean)", np.tanh(pol.mu), kw)
            r_bs, _, a_bs = report("best sampled", best[1], kw)
            a = a_mu if r_mu >= r_bs else a_bs

            np.save(fname + ".npy", a)
            np.savez(fname + "_meta.npz", action=a, plant=plant, mode=tag,
                     fuzzy_type=ft, tune_fou=tune, c0x2=1, seed=seed,
                     reward=max(r_mu, r_bs), hist=hist, mu=pol.mu, logstd=pol.logstd,
                     n_iters=N_ITERS, epochs=EPOCHS, G=G)
            env0.describe(a)
            print(f"saved -> {fname}.npy   ({(time.time() - t0) / 60:.1f} min)")

### 4b. Training with ±15% plant variation
Files: `a_fis_{plant}_{tag}_dr15_seed{seed}.npy`

In [ ]:
import time
import multiprocessing as mp
from grpo_workers import init_worker, rollout_on

SEEDS     = [0, 1, 2, 3, 4]
N_WORKERS = len(cpus)
RUNS = [('it2', 2, False), ('t1', 1, False), ('fou', 2, True)]

def make_kw(plant, ft, tune):
    return {**env_kw_for(plant), 'fuzzy_type': ft, 'tune_fou': tune}

def run_file(plant, tag, seed):
    return f"a_fis_{plant}_{tag}_dr15_seed{seed}"

def is_done(fname):
    try:
        meta = np.load(fname + "_meta.npz", allow_pickle=True)
        return int(meta['c0x2']) == 1 and len(meta['hist']) == N_ITERS
    except Exception:
        return False

print(f"N_ITERS = {N_ITERS}, inner EPOCHS = {EPOCHS}, G = {G}, DELTA = {DELTA}")
check_c0_fix()
ctx = mp.get_context('spawn')

for seed in SEEDS:
    for tag, ft, tune in RUNS:
        for plant in PLANTS_TO_RUN:
            fname = run_file(plant, tag, seed)
            if is_done(fname):
                print(f"[{tag} {plant} seed {seed}] done, skipping")
                continue

            print(f"\n=== {tag}: {PLANTS[plant]['name']}, seed {seed} ===")
            kw = make_kw(plant, ft, tune)
            env0 = make_env(**kw)
            t0 = time.time()

            with ctx.Pool(processes=N_WORKERS, initializer=init_worker,
                          initargs=(kw,)) as pool:
                a0 = env0.nominal_action()
                assert abs(pool.map(rollout_on, [(a0, DELTA, None)])[0]
                           - env0.rollout(a0)) < 1e-9
                pol, hist, val_hist, a, r_val, best_it = train_dr(
                    N_ITERS, kw, seed, pool)

            report(f"best val @{best_it}", a, kw)
            np.save(fname + ".npy", a)
            np.savez(fname + "_meta.npz", action=a, plant=plant, mode=tag,
                     fuzzy_type=ft, tune_fou=tune, c0x2=1, seed=seed,
                     reward=r_val, best_it=best_it, hist=hist, val_hist=val_hist,
                     mu=pol.mu, logstd=pol.logstd, delta=DELTA,
                     n_iters=N_ITERS, epochs=EPOCHS, G=G)
            env0.describe(a)
            print(f"saved -> {fname}.npy   ({(time.time() - t0) / 60:.1f} min)")

###### Sweeep

## 5. Baselines

Same plants, decoupler, IMC-PID starting point, input limits, derivative
filter, GRPO settings and budget as section 4.
* `su` – Sahin & Ulu (2026), SI-IT2-FPID with online FOU tuning: PID gains +
  k1, k2 per loop. Trained on the nominal plant and with ±15% variation.
* `kz` – Kluska & Zabinski (2020), PID-like fuzzy controller: stability sector
  (Theorem 1), FC-2 consequents tuned on the nominal plant (Step 3), then the
  adaptation gain picked on the validation plants for both starting points
  (`gamma_pid`, `gamma_ref` in the meta file).

Files: `a_fis_{plant}_su_seed{seed}`, `a_fis_{plant}_su_dr15_seed{seed}`,
`a_fis_{plant}_kz_seed{seed}`.

In [ ]:
import os, time
import multiprocessing as mp
import numpy as np
from mimo_fuzzy_env import PLANTS
from baselines import make_env, kluska_select_gamma
from paper_designs import design_sahin_ulu, design_kluska

N_WORKERS    = len(cpus)
KZ_CRITERION = 'paper'        # 'paper' (no overshoot, shortest settling) | 'iae'

def done(fname):
    return os.path.exists(fname + "_meta.npz")

ctx = mp.get_context('spawn')
with ctx.Pool(processes=N_WORKERS) as pool:
    for plant in PLANTS_TO_RUN:
        kw = env_kw_for(plant)

        # ---------------- Sahin & Ulu
        fname = f"a_fis_{plant}_su_paper"
        if done(fname):
            print(f"[{fname}] done, skipping")
        else:
            print(f"\n=== Sahin & Ulu: {PLANTS[plant]['name']} ===")
            t0 = time.time()
            a, log = design_sahin_ulu(kw, pool)
            np.save(fname + ".npy", a)
            np.savez(fname + "_meta.npz", action=a, plant=plant,
                     method='sahin_ulu', design='paper', **log)
            make_env(**{**kw, 'method': 'sahin_ulu', 'tune_pid': False}).describe(a)
            print(f"  IAE: start {log['iae_pid_start']:.3f} -> constant FOU "
                  f"{log['iae_const_fou']:.3f} -> OTM {log['iae_otm']:.3f}")
            print(f"saved -> {fname}.npy   ({(time.time() - t0) / 60:.1f} min)")

        # ---------------- Kluska & Zabinski
        fname = f"a_fis_{plant}_kz_paper"
        if done(fname):
            print(f"[{fname}] done, skipping")
        else:
            print(f"\n=== Kluska & Zabinski: {PLANTS[plant]['name']} ===")
            t0 = time.time()
            a, log = design_kluska(kw, pool, criterion=KZ_CRITERION)
            # Step 5: adaptation gain on the nominal plant only, start = linear PID
            c, scores = kluska_select_gamma(kw, a, q_init='pid', val_seeds=())
            log['gamma_pid'] = c
            print(f"  gamma factor: {c}   ("
                  + ", ".join(f"{k}: {v:.2f}" for k, v in scores.items()) + ")")
            np.save(fname + ".npy", a)
            np.savez(fname + "_meta.npz", action=a, plant=plant, method='kluska',
                     design='paper', criterion=KZ_CRITERION, **log)
            make_env(**{**kw, 'method': 'kluska'}).describe(a)
            print(f"saved -> {fname}.npy   ({(time.time() - t0) / 60:.1f} min)")

## 6. Compare all methods
Mean ± std over seeds of IAE and ISE on the nominal, +30% and −30% plants.
Missing files are reported as "no files", so this can run at any time.
Set `SHOW_SEEDS = True` in `eval_compare.py` for one row per seed.
All values are also written to `results_all.csv`.

In [ ]:
%run eval_compare.py

#### Fuzzy-IMC values

In [ ]:
from mimo_fuzzy_env import MIMOPlant, PLANTS
from baselines import make_env, perturb

for plant in ('vl', 'wb', 'ww', 'isp', 'hvac'):
    env = make_env(**{**env_kw_for(plant), 'fuzzy_type': 2, 'tune_fou': False})
    nom, cells = env.plant, []
    for p in (0.0, 0.30, -0.30):
        env.plant = nom if p == 0 else MIMOPlant(perturb(env.spec, p), env.ts)
        m = env.metrics(env.nominal_action())
        cells += [f"{m['iae_tot']:.2f}", f"{m['ise_tot']:.2f}"]
    print(plant, ' & '.join(cells))